# CAROM prototyping ladder: scheduled / fixed-point / itinerant
Toy-scale prototype of three execution semantics over one shared operator library
(command-only softmax routing over K read-transform-gate-write operators on a
slot workspace). Task: composing whole-sequence transforms over Z_8
(inc, dbl, neg, reverse, rotate, swap, prefix-sum + 2 synonym tokens + noop padding).

**Pre-resolved kinks (found in a CPU sandbox dry run -- keep these!):**
1. **Routing bottleneck**: command-only routing realizes ~K distinct primitive behaviors
   (simplex vertices + blends). Keep command vocab small & behavior-typed; slot-addressed
   instruction formats (e.g. `slot5=add(slot2,slot3)`) explode the needed K and stall learning.
2. **Inhibition convention**: with `comp_m = sum_j rho[m,j] a_j`, `rho[m,j]` = inhibition
   *felt by m from j*. Chain: successor feels ~0.35 from predecessor (invadable);
   predecessor feels ~4.0 from successor (extinguishable); all else ~2.5; self 1.0.
3. **Fitness baseline**: positive base (~3.0) or total control activity collapses.
4. **Channel entry**: init a_0 = 1.0, others at floor. Uniform init structurally favors
   middle modes (each successor gets an inhibition discount; mode 0 gets none).
5. **Activation floor 0.05** (not 1e-4): saddle passage time ~ ln(1/floor)/lambda.
6. **Zero-init per-command fitness biases** -> untrained channel is deterministic (64/64).
7. **DEQ contraction penalty**: without `+ resid[:,-1].mean()` in the loss, the
   fixed-point model silently trains into an unrolled net with *growing* residuals.
   Always plot residual-per-sweep.

Sandbox reference numbers (CPU, d=48, small budgets -- expect better on GPU):
scheduled ~70% @4k steps (chance 12.5%); fixed-point depth-graded 0.44/0.33/0.29/0.19 @~0.8k;
itinerant (hand-built chain) 256/256 sequential itineraries preserved after training,
acc above chance and climbing @~0.45k; free-rho needs longer training + likely an
explicit multi-phase pressure (open experiment).

In [ ]:
import torch, random, math
import torch.nn as nn, torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# GPU preset vs CPU preset
GPU = DEVICE == "cuda"
D_MODEL   = 64 if GPU else 48
STEPS_R1  = 8000 if GPU else 4000
STEPS_R3  = 6000 if GPU else 1400
STEPS_R45 = 4000 if GPU else 500
BS        = 256 if GPU else 128
print(DEVICE)

In [ ]:
# %% Task: sequence-transform composition programs
"""Sequence-transform composition task for CAROM prototyping.

Workspace: n = 6 slots over Z_8.  Commands are whole-sequence transforms
(small vocab -- compatible with CAROM's K-operator routing bottleneck).
A program is up to L_MAX commands, executed in presentation order; shorter
programs are padded with trailing 'noop'.  Two synonym tokens force routing
reuse.  Critical-path depth of a sample = number of non-noop commands.
"""
import random
import torch

V = 8
N_SLOTS = 6
L_MAX = 5

def _inc(v): return [(x + 1) % V for x in v]
def _dbl(v): return [(2 * x) % V for x in v]
def _neg(v): return [(-x) % V for x in v]
def _rev(v): return v[::-1]
def _shr(v): return [v[-1]] + v[:-1]
def _swp(v): return [v[1], v[0]] + v[2:]
def _cms(v):
    out, s = [], 0
    for x in v: s = (s + x) % V; out.append(s)
    return out
def _nop(v): return v

OPS = [("noop", _nop), ("inc", _inc), ("dbl", _dbl), ("neg", _neg),
       ("rev", _rev), ("shr", _shr), ("swp", _swp), ("cms", _cms),
       ("inc_syn", _inc), ("rev_syn", _rev)]          # last two = synonyms
C_VOCAB = len(OPS)
NOOP = 0
REAL = list(range(1, C_VOCAB))

def make_batch(bs, rng, device=None, fixed_len=None):
    device = device or DEVICE
    C = torch.zeros(bs, L_MAX, dtype=torch.long)
    X = torch.zeros(bs, N_SLOTS, dtype=torch.long)
    Y = torch.zeros(bs, N_SLOTS, dtype=torch.long)
    D = torch.zeros(bs, dtype=torch.long)             # effective program length
    for b in range(bs):
        L = fixed_len if fixed_len else rng.randint(2, L_MAX)
        cmds = [rng.choice(REAL) for _ in range(L)] + [NOOP] * (L_MAX - L)
        vals = [rng.randrange(V) for _ in range(N_SLOTS)]
        out = list(vals)
        for c in cmds:
            out = OPS[c][1](out)
        C[b] = torch.tensor(cmds); X[b] = torch.tensor(vals)
        Y[b] = torch.tensor(out); D[b] = L
    return C.to(device), None, X.to(device), Y.to(device), D.to(device)



In [ ]:
# %% Models: shared operator core + three execution semantics
"""Shared operator core + three execution semantics."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class OperatorCore(nn.Module):
    """K parallel read->transform->gate->write operators, batched along K.

    forward(w, feat, extra) -> u of shape [B(, M), K, n, d]
      w    : [..., n, d]   mutable content
      feat : [..., n, 2d]  concat(evidence, position), constant per sample
      extra: [..., 1, d] or None  additive query/key conditioning (step emb)
    """
    def __init__(self, d, K):
        super().__init__()
        self.d, self.K = d, K
        def pk(*shape):
            t = torch.empty(*shape); nn.init.xavier_uniform_(t.view(shape[0], -1))
            return nn.Parameter(t)
        self.Wq = pk(K, 3 * d, d); self.Wk = pk(K, 3 * d, d); self.Wv = pk(K, 3 * d, d)
        self.T1 = pk(K, 4 * d, 2 * d); self.T2 = pk(K, 2 * d, d)
        self.Wg = pk(K, d, d)
        self.gb = nn.Parameter(torch.zeros(K, d))            # gate bias ~0 -> gate ~0.5
        self.ln = nn.LayerNorm(d)

    def forward(self, w, feat, extra=None):
        h = torch.cat([self.ln(w), feat], dim=-1)            # [..., n, 3d]
        if extra is not None:
            h = h + F.pad(extra, (0, 2 * self.d))            # condition content channel
        q = torch.einsum("...nf,kfd->...knd", h, self.Wq)
        k = torch.einsum("...nf,kfd->...knd", h, self.Wk)
        v = torch.einsum("...nf,kfd->...knd", h, self.Wv)
        att = torch.einsum("...knd,...kmd->...knm", q, k) / math.sqrt(self.d)
        r = torch.einsum("...knm,...kmd->...knd", att.softmax(-1), v)
        z = torch.cat([r, self.ln(w).unsqueeze(-3).expand_as(r),
                       feat.unsqueeze(-3).expand(*r.shape[:-1], 2 * self.d)], dim=-1)
        z = torch.einsum("...knf,kfe->...kne", F.gelu(
            torch.einsum("...knf,kfe->...kne", z, self.T1)), self.T2)
        g = torch.sigmoid(torch.einsum("...knd,kde->...kne", z, self.Wg)
                          + self.gb.view(*([1] * (z.dim() - 3)), self.K, 1, self.d))
        return g * torch.tanh(z)                              # u

class Base(nn.Module):
    def __init__(self, d=48, K=8):
        super().__init__()
        self.d, self.K = d, K
        self.sym = nn.Embedding(V + 1, d)                     # +1 blank
        self.pos = nn.Embedding(N_SLOTS, d)
        self.route = nn.Embedding(C_VOCAB, K)                 # routing table A
        self.core = OperatorCore(d, K)
        self.out = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2 * d),
                                 nn.GELU(), nn.Linear(2 * d, V))
        self.tau = 1.0

    def embed(self, X):
        e = F.normalize(self.sym(X), dim=-1) * math.sqrt(self.d)
        p = self.pos(torch.arange(N_SLOTS, device=X.device)).expand_as(e)
        return e, torch.cat([e, p], dim=-1)                   # w0, feat

    def rho(self, C):                                         # [B, M, K]
        return F.softmax(self.route(C) / self.tau, dim=-1)

    def readout(self, w):
        return self.out(w)                                    # [B, n, V]

class Scheduled(Base):
    def forward(self, C, X):
        w, feat = self.embed(X)
        beta = self.rho(C)                                    # [B, L, K]
        for t in range(C.shape[1]):
            u = self.core(w, feat)                            # [B, K, n, d]
            w = w + torch.einsum("bk,bknd->bnd", beta[:, t], u)
        return self.readout(w)

class FixedPoint(Base):
    def __init__(self, d=48, K=8, sweeps=8, alpha=0.5):
        super().__init__(d, K)
        self.sweeps, self.alpha = sweeps, alpha
        self.step = nn.Embedding(L_MAX, d)                    # step embedding s_m

    def field(self, w, feat, C, P):
        s = self.step(P).unsqueeze(2)                         # [B, M, 1, d]
        u = self.core(w.unsqueeze(1), feat.unsqueeze(1), extra=s)   # [B, M, K, n, d]
        beta = self.rho(C)                                    # [B, M, K]
        return w + torch.einsum("bmk,bmknd->bnd", beta, u) / C.shape[1]

    def forward(self, C, X, P=None, return_resid=False):
        w, feat = self.embed(X)
        if P is None:
            P = torch.arange(C.shape[1], device=C.device).expand(C.shape[0], -1)
        resid = []
        for _ in range(self.sweeps):
            wn = (1 - self.alpha) * w + self.alpha * self.field(w, feat, C, P)
            resid.append((wn - w).norm(dim=(-2, -1)) / w.norm(dim=(-2, -1)))
            w = wn
        out = self.readout(w)
        return (out, torch.stack(resid, 1)) if return_resid else out

class Itinerant(Base):
    def __init__(self, d=48, K=8, steps=70, dt=0.2, noise=0.02,
                 fatigue_tau=6.0, fatigue_k=1.5, leak=0.02, fixed_chain=False):
        super().__init__(d, K)
        self.S, self.dt, self.noise = steps, dt, noise
        self.ftau, self.fk, self.leak = fatigue_tau, fatigue_k, leak
        self.M = L_MAX
        self.sigma = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(),
                                   nn.Linear(d, 1))
        self.sig_cmd = nn.Embedding(C_VOCAB, 1)               # per-token fitness bias
        nn.init.zeros_(self.sig_cmd.weight)
        self.sig_scale = nn.Parameter(torch.tensor(0.5))
        self.base, self.ord = 3.0, 0.1                       # tuned below
        self.pool_q = nn.Parameter(torch.randn(d) / math.sqrt(d))
        rho0 = torch.full((self.M, self.M), 2.5)              # strong mutual inhibition
        if fixed_chain:                                       # hand-built successor chain
            for m in range(self.M):
                rho0[m, m] = 1.0
                if m + 1 < self.M:
                    rho0[m + 1, m] = 0.35                     # successor feels little from m -> can invade
                    rho0[m, m + 1] = 4.0                      # m feels much from successor -> dies
        else:
            rho0 += 0.1 * torch.randn(self.M, self.M)
            rho0.fill_diagonal_(1.0)
        self.rho_raw = nn.Parameter(torch.log(torch.expm1(rho0)))  # inv-softplus
        self.rho_raw.requires_grad = not fixed_chain

    def fitness(self, w, C, P=None):
        if P is None:
            P = torch.arange(C.shape[1], device=C.device).expand(C.shape[0], -1)
        attn = torch.einsum("bnd,d->bn", w, self.pool_q).softmax(-1)
        pooled = torch.einsum("bn,bnd->bd", attn, w)
        learned = self.sigma(pooled) + self.sig_cmd(C).squeeze(-1)
        return self.base + self.sig_scale * learned - self.ord * P.float()
    def forward(self, C, X, P=None, return_traj=False):
        B = C.shape[0]; dev = C.device
        w, feat = self.embed(X)
        a = torch.full((B, self.M), 0.05, device=dev)
        a[:, 0] = 1.0                                          # enter channel at station 0
        a = a + 0.01 * torch.randn_like(a).abs()
        f = torch.zeros_like(a)
        rho_m = F.softplus(self.rho_raw)
        beta_cmd = self.rho(C)                                # [B, M, K]
        traj = []
        for t in range(self.S):
            fit = self.fitness(w, C, P) - self.fk * f
            comp = torch.einsum("mj,bj->bm", rho_m, a)
            noise = self.noise * torch.randn_like(a) if self.training else 0.0
            a = (a + self.dt * a * (fit - comp) + noise).clamp(0.05, 4.0)
            f = f + self.dt / self.ftau * (a - f)
            beta = torch.einsum("bm,bmk->bk", a, beta_cmd)    # [B, K]
            u = self.core(w, feat)
            w = w + self.dt * (torch.einsum("bk,bknd->bnd", beta, u) - self.leak * w)
            if return_traj:
                traj.append(a.detach().clone())
        out = self.readout(w)
        return (out, torch.stack(traj, 1)) if return_traj else out


In [ ]:
# %% Training harness (checkpoint/resume, time-budgeted)
import random, time
import torch, torch.nn.functional as F

def accuracy(logits, Y):
    return (logits.argmax(-1) == Y).float().mean().item()

def run(variant, steps=1500, bs=128, lr=2e-3, scrambled=False, seed=0,
        ckpt=None, budget=280, **kw):
    torch.manual_seed(seed); rng = random.Random(seed)
    model = {"scheduled": Scheduled, "fixedpoint": FixedPoint,
             "itinerant": Itinerant}[variant](**kw).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    start = 0
    if ckpt:
        try:
            st = torch.load(ckpt, weights_only=False)
            model.load_state_dict(st["m"]); opt.load_state_dict(st["o"])
            sched.load_state_dict(st["s"]); start = st["it"] + 1
            rng.setstate(st["rng"]); torch.set_rng_state(st["trng"])
            print(f"  resumed at it {start}")
        except FileNotFoundError:
            pass
    t0 = time.time()
    acc = 0.0
    for it in range(start, steps):
        if time.time() - t0 > budget:
            break
        C, P, X, Y, D = make_batch(bs, rng)
        if variant == "fixedpoint":
            logits, resid = model(C, X, return_resid=True)
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), Y.reshape(-1)) \
                   + 1.0 * resid[:, -1].mean()          # contraction pressure
        else:
            logits = model(C, X)
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), Y.reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if it % 150 == 0 or it == steps - 1:
            model.eval()
            with torch.no_grad():
                C, P, X, Y, D = make_batch(512, rng)
                logits = model(C, X)
                acc = accuracy(logits, Y)
            model.train()
            print(f"  it {it:4d}  loss {loss.item():.3f}  eval-acc {acc:.3f}"
                  f"  ({time.time()-t0:.0f}s)", flush=True)
    if ckpt:
        torch.save({"m": model.state_dict(), "o": opt.state_dict(),
                    "s": sched.state_dict(), "it": it, "rng": rng.getstate(),
                    "trng": torch.get_rng_state()}, ckpt)
        print(f"  saved {ckpt} at it {it}")
    return model, acc



## Rung 0 -- generator self-check

In [ ]:
rng = random.Random(0)
C, P, X, Y, D = make_batch(4, rng)
v = X[2].tolist()
for c in C[2].tolist(): v = OPS[c][1](v)
assert v == Y[2].tolist(); print("generator OK; vocab", C_VOCAB)

## Rung 1 -- scheduled CAROM (sanity + baseline)

In [ ]:
model_r1, acc_r1 = run("scheduled", steps=STEPS_R1, bs=BS, ckpt="r1.pt",
                        budget=10**9, d=D_MODEL)

## Rung 3 -- fixed-point CAROM
Watch the residual plot: it must *decay* across sweeps. If it grows, the contraction penalty is missing/too weak.

In [ ]:
model_r3, acc_r3 = run("fixedpoint", steps=STEPS_R3, bs=BS, ckpt="r3.pt",
                        budget=10**9, d=D_MODEL)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
model_r3.eval(); rng = random.Random(99)
with torch.no_grad():
    for L in [2,3,4,5]:
        C,P,X,Y,Dd = make_batch(512, rng, fixed_len=L)
        out, resid = model_r3(C, X, return_resid=True)
        print(f"depth {L}: acc {(out.argmax(-1)==Y).float().mean():.3f}")
        plt.plot(resid.mean(0).cpu(), label=f"depth {L}")
plt.xlabel("sweep"); plt.ylabel("rel. residual"); plt.legend(); plt.title("must decay!"); plt.show()

## Rung 4 -- itinerant CAROM, hand-built chain (mechanism isolation)

In [ ]:
model_r4, acc_r4 = run("itinerant", steps=STEPS_R45, bs=BS//2, ckpt="r4.pt",
                        budget=10**9, d=D_MODEL, fixed_chain=True)

In [ ]:
def phases(a):
    s = a.argmax(-1).tolist(); out=[s[0]]
    for x in s[1:]:
        if x != out[-1]: out.append(x)
    return out
model_r4.eval(); rng = random.Random(7)
with torch.no_grad():
    C,P,X,Y,Dd = make_batch(256, rng, fixed_len=5)
    out, traj = model_r4(C, X, return_traj=True)
print("acc", round((out.argmax(-1)==Y).float().mean().item(),3),
      " perfect itineraries:", sum(phases(traj[b])==[0,1,2,3,4] for b in range(256)), "/256")
import matplotlib.pyplot as plt
plt.stackplot(range(traj.shape[1]), traj[0].T.cpu(), alpha=.85,
              labels=[f"mode {m}" for m in range(5)])
plt.legend(); plt.xlabel("t"); plt.ylabel("a_m(t)"); plt.title("sequential handoffs"); plt.show()

## Rung 5 -- free rho (the open experiment)
Order must be *discovered*. Sandbox finding: 124 steps is nowhere near enough
(single-phase winner-take-all, tau ~ 0). Suggested pressures to add if it stalls:
(a) itinerary/entropy regularizer encouraging >=L phase transitions;
(b) anneal fatigue k upward; (c) curriculum from short programs;
(d) initialize rho from a *weakly* asymmetric prior and let learning sharpen it.

In [ ]:
model_r5, acc_r5 = run("itinerant", steps=STEPS_R45, bs=BS//2, ckpt="r5.pt",
                        budget=10**9, d=D_MODEL, fixed_chain=False)

In [ ]:
def kendall(seq):
    if len(seq) < 2: return 0.0
    c = sum(seq[i] < seq[j] for i in range(len(seq)) for j in range(i+1, len(seq)))
    d = sum(seq[i] > seq[j] for i in range(len(seq)) for j in range(i+1, len(seq)))
    return (c-d)/max(1, c+d)
model_r5.eval()
with torch.no_grad():
    C,P,X,Y,Dd = make_batch(256, rng, fixed_len=5)
    out, traj = model_r5(C, X, return_traj=True)
rho = F.softplus(model_r5.rho_raw).detach().cpu()
print("acc", round((out.argmax(-1)==Y).float().mean().item(),3),
      " asym", round((rho-rho.T).norm().item(),3),
      " mean tau", round(np.mean([kendall(phases(traj[b])) for b in range(256)]),3),
      " mean phases", round(np.mean([len(phases(traj[b])) for b in range(256)]),2))